# Model Training & Evaluation

Implements Design Doc §5.3-§5.5 and [IMPLEMENTATION_PLAN.md](../IMPLEMENTATION_PLAN.md) Phase 4: scaffold split, XGBoost-on-ECFP vs. MLP-on-ChemBERTa, evaluated with RMSE/R²/Spearman.

This notebook is being built incrementally, one plan step at a time. **This pass covers steps 1-6: the scaffold split, training both models, the formal held-out evaluation, a direct (bootstrap-backed) comparison, and saving models/metrics to `results/`.** Diagnostic plots (step 7) come in a later pass.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append("../src")
from data_utils import scaffold_split

PROCESSED_DIR = Path("../data/processed")

df = pd.read_csv(PROCESSED_DIR / "kit_bioactivity_clean.csv")
print("Loaded:", df.shape)
df.head()

Loaded: (5565, 10)


,molecule_chembl_id,kit_variant,canonical_smiles,p_value,censored,censored_direction,n_measurements,p_value_std,n_documents,standard_types
0,CHEMBL10,D816V,C[S+]([O-])c1ccc(-c2nc(-c3ccc(F)cc3)c(-c3ccncc...,5.000000,True,>,3.0,NaN,3.0,Kd
1,CHEMBL10,WT,C[S+]([O-])c1ccc(-c2nc(-c3ccc(F)cc3)c(-c3ccncc...,5.000000,True,>,13.0,NaN,4.0,"Kd,Ki"
2,CHEMBL101253,D816V,Clc1ccc(Nc2nnc(Cc3ccncc3)c3ccccc23)cc1,5.000000,True,>,3.0,NaN,3.0,Kd
3,CHEMBL101253,WT,Clc1ccc(Nc2nnc(Cc3ccncc3)c3ccccc23)cc1,6.677781,False,NaN,17.0,1.308074,7.0,"IC50,Kd,Ki"
4,CHEMBL101683,WT,O=C(Nc1ccc(Cl)cc1)c1ccccc1NCc1ccncc1,6.619789,False,NaN,1.0,NaN,1.0,IC50


## 1. Scaffold split (train/test)

Design Doc §5.4: use a **scaffold split**, not a random split — grouping compounds by Bemis-Murcko scaffold before splitting, since random splits let near-identical analogues leak across train/test and overestimate generalization.

`scaffold_split` (in [`src/data_utils.py`](../src/data_utils.py)) groups row indices by scaffold, then assigns whole scaffold groups — largest first — to train until an 80% target is hit, with the remainder (smaller, rarer-scaffold groups) going to test. WT/D816V rows of the same compound share identical SMILES and therefore identical scaffolds, so they always land on the same side of the split; no special-casing needed.

In [2]:
train_idx, test_idx = scaffold_split(df["canonical_smiles"].tolist(), frac_train=0.8, seed=0)

print(f"Train: {len(train_idx)} rows ({len(train_idx) / len(df):.1%})")
print(f"Test:  {len(test_idx)} rows ({len(test_idx) / len(df):.1%})")

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

Train: 4452 rows (80.0%)
Test:  1113 rows (20.0%)


### Verify no scaffold leakage between train and test

The whole point of a scaffold split is that no scaffold appears on both sides — confirm that directly rather than trusting the split logic didn't have an off-by-one.

In [3]:
from data_utils import bemis_murcko_scaffold

train_scaffolds = set(train_df["canonical_smiles"].map(bemis_murcko_scaffold))
test_scaffolds = set(test_df["canonical_smiles"].map(bemis_murcko_scaffold))

overlap = train_scaffolds & test_scaffolds
print(f"Unique scaffolds — train: {len(train_scaffolds)}, test: {len(test_scaffolds)}")
print(f"Scaffolds appearing in both: {len(overlap)}")
assert not overlap, "Scaffold leakage between train and test!"

# Sanity-check the WT/D816V co-location invariant: every compound's rows should
# be entirely in train or entirely in test, never split across both.
split_side = pd.Series("train", index=df.index)
split_side.iloc[test_idx] = "test"
sides_per_compound = df.groupby("molecule_chembl_id").apply(
    lambda g: split_side.loc[g.index].nunique(), include_groups=False
)
straddling = sides_per_compound[sides_per_compound > 1]
print(f"Compounds with rows split across train AND test: {len(straddling)}")
assert straddling.empty, "A compound's WT/D816V rows ended up on different sides of the split!"


Unique scaffolds — train: 814, test: 1113
Scaffolds appearing in both: 0
Compounds with rows split across train AND test: 0


## 2. Baseline model: XGBoost on ECFP fingerprints

Design Doc §5.3: gradient-boosted trees on fingerprint features are a realistic, strong baseline given the likely small-to-medium dataset size, and shouldn't be skipped in favor of jumping straight to the ChemBERTa-based model.

`train_xgboost_ecfp` (in [`src/models.py`](../src/models.py)) trains on the cached ECFP fingerprints from `03_featurization.ipynb`, sliced to `train_idx`/`test_idx` from the scaffold split above. **This step only trains the model and sanity-checks that it actually fit** — the formal held-out RMSE/R²/Spearman evaluation (Phase 4 step 4) comes once the ChemBERTa/MLP model exists too, so both can be compared side by side.

In [4]:
ecfp = np.load(PROCESSED_DIR / "ecfp_fingerprints.npy")
assert ecfp.shape[0] == len(df), "ECFP cache is not row-aligned with the cleaned dataset"

y = df["p_value"].to_numpy()

X_train, X_test = ecfp[train_idx], ecfp[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print("X_train:", X_train.shape, " y_train:", y_train.shape)
print("X_test: ", X_test.shape, " y_test: ", y_test.shape)

X_train: (4452, 2048)  y_train: (4452,)
X_test:  (1113, 2048)  y_test:  (1113,)


In [5]:
from models import train_xgboost_ecfp

xgb_ecfp = train_xgboost_ecfp(X_train, y_train)
print(f"Trained XGBRegressor on {X_train.shape[0]} compounds, {X_train.shape[1]}-bit ECFP features.")

Trained XGBRegressor on 4452 compounds, 2048-bit ECFP features.


### Sanity-check: did it actually fit?

Not the formal evaluation (that's step 4) — just confirming the model learned a real, non-degenerate relationship rather than silently failing (e.g. from a features/labels misalignment) before moving on. Tree ensembles should fit training data closely; a near-zero training R² would mean something upstream is broken. A quick look at held-out predictions checks they aren't constant and are at least positively correlated with truth.

In [6]:
from sklearn.metrics import r2_score

train_r2 = r2_score(y_train, xgb_ecfp.predict(X_train))
print(f"Training-set R² (fit sanity check, NOT held-out performance): {train_r2:.3f}")
assert train_r2 > 0.5, "Model barely fit the training data -- check feature/label alignment"

test_preds = xgb_ecfp.predict(X_test)
test_corr = np.corrcoef(test_preds, y_test)[0, 1]
print(f"Held-out prediction std: {test_preds.std():.3f} (non-degenerate: not a constant prediction)")
print(f"Held-out Pearson correlation (predicted vs. actual p_value): {test_corr:.3f}")
assert test_preds.std() > 1e-3, "Model is predicting a near-constant value on held-out data"
assert test_corr > 0, "Held-out predictions are not even positively correlated with truth"

Training-set R² (fit sanity check, NOT held-out performance): 0.809
Held-out prediction std: 0.841 (non-degenerate: not a constant prediction)
Held-out Pearson correlation (predicted vs. actual p_value): 0.725


## 3. Comparison model: small MLP head on frozen ChemBERTa embeddings

Design Doc §5.3: a small MLP on top of the frozen ChemBERTa embeddings from `03_featurization.ipynb`, compared directly against the XGBoost-on-ECFP baseline above rather than assumed to win — echoing the dissertation's own finding that a simpler method can outperform a more sophisticated one on a small dataset.

`train_mlp_chemberta` (in [`src/models.py`](../src/models.py)) standardizes the embeddings first (MLP training is scale-sensitive, unlike the tree-based baseline) and uses early stopping so training doesn't run longer than needed. **Same scope as step 2: train + sanity-check only** — the formal head-to-head comparison is step 4.

In [7]:
chemberta = np.load(PROCESSED_DIR / "chemberta_embeddings.npy")
assert chemberta.shape[0] == len(df), "ChemBERTa cache is not row-aligned with the cleaned dataset"

X_train_cb, X_test_cb = chemberta[train_idx], chemberta[test_idx]
# y_train / y_test (the p_value labels) are already defined from step 2 above --
# same rows, same split, just a different feature matrix.

print("X_train_cb:", X_train_cb.shape, " y_train:", y_train.shape)
print("X_test_cb: ", X_test_cb.shape, " y_test: ", y_test.shape)

X_train_cb: (4452, 768)  y_train: (4452,)
X_test_cb:  (1113, 768)  y_test:  (1113,)


In [8]:
from models import train_mlp_chemberta

mlp_chemberta = train_mlp_chemberta(X_train_cb, y_train)
n_iter = mlp_chemberta.named_steps["mlpregressor"].n_iter_
print(f"Trained MLP on {X_train_cb.shape[0]} compounds, 768-dim ChemBERTa embeddings ({n_iter} iterations, early-stopped).")

Trained MLP on 4452 compounds, 768-dim ChemBERTa embeddings (42 iterations, early-stopped).


### Sanity-check: did it actually fit?

Same reasoning as the XGBoost sanity check above — confirm the model learned a real relationship before moving on, not the formal evaluation.

In [9]:
train_r2_mlp = r2_score(y_train, mlp_chemberta.predict(X_train_cb))
print(f"Training-set R² (fit sanity check, NOT held-out performance): {train_r2_mlp:.3f}")
assert train_r2_mlp > 0.5, "Model barely fit the training data -- check feature/label alignment"

test_preds_mlp = mlp_chemberta.predict(X_test_cb)
test_corr_mlp = np.corrcoef(test_preds_mlp, y_test)[0, 1]
print(f"Held-out prediction std: {test_preds_mlp.std():.3f} (non-degenerate: not a constant prediction)")
print(f"Held-out Pearson correlation (predicted vs. actual p_value): {test_corr_mlp:.3f}")
assert test_preds_mlp.std() > 1e-3, "Model is predicting a near-constant value on held-out data"
assert test_corr_mlp > 0, "Held-out predictions are not even positively correlated with truth"

Training-set R² (fit sanity check, NOT held-out performance): 0.748
Held-out prediction std: 1.174 (non-degenerate: not a constant prediction)
Held-out Pearson correlation (predicted vs. actual p_value): 0.562


## 4. Evaluate both models on the held-out set

Design Doc §5.5: RMSE and R² on held-out `p_value`, plus Spearman rank correlation as a secondary metric — for potency prediction, getting the *relative ranking* of compounds right often matters more in practice than the exact value.

`evaluate_regression` (in [`src/evaluation.py`](../src/evaluation.py)) computes all three from `(y_true, y_pred)`. Applied here to both models' held-out predictions from steps 2-3, on the identical 1,113-row scaffold-split test set. (Directly comparing the two — which wins and why — is step 5; this step is just computing the numbers.)

In [10]:
from evaluation import evaluate_regression

ecfp_metrics = evaluate_regression(y_test, xgb_ecfp.predict(X_test))
chemberta_metrics = evaluate_regression(y_test, mlp_chemberta.predict(X_test_cb))

metrics_df = pd.DataFrame(
    {"XGBoost + ECFP": ecfp_metrics, "MLP + ChemBERTa": chemberta_metrics}
).T
metrics_df.columns = ["RMSE", "R²", "Spearman ρ"]
metrics_df.round(3)

,RMSE,R²,Spearman ρ
XGBoost + ECFP,0.849,0.520,0.702
MLP + ChemBERTa,1.125,0.156,0.551


### Sanity-check against the naive baseline

Both models should clearly beat "always predict the training-set mean" — otherwise they aren't capturing any real structure-activity signal at all.

In [11]:
naive_baseline_pred = np.full_like(y_test, y_train.mean())
naive_rmse = np.sqrt(np.mean((y_test - naive_baseline_pred) ** 2))
print(f"Naive mean-baseline held-out RMSE: {naive_rmse:.3f} (R²=0 by construction)")
# (Spearman is undefined for a constant prediction, so skip it here rather
# than calling evaluate_regression and triggering a ConstantInputWarning for
# a value we don't need.)

for name, metrics in [("XGBoost + ECFP", ecfp_metrics), ("MLP + ChemBERTa", chemberta_metrics)]:
    assert metrics["rmse"] < naive_rmse, f"{name} failed to beat the naive mean baseline on RMSE"
    assert metrics["r2"] > 0, f"{name} has non-positive R² -- no better than predicting the mean"
print("Both models beat the naive mean-baseline on RMSE and R².")

Naive mean-baseline held-out RMSE: 1.233 (R²=0 by construction)
Both models beat the naive mean-baseline on RMSE and R².


## 5. Compare the two models directly

Design Doc §5.3: compare baseline vs. embedding model directly — don't assume the more sophisticated model wins, echoing the dissertation's own GAN-vs-augmentation finding that a simpler method can beat a fancier one on a small dataset. Step 4's single point estimate already shows XGBoost + ECFP ahead on every metric, but a single fixed 1,113-row test set could make a real gap look bigger (or smaller) than it robustly is. `bootstrap_compare` (in [`src/evaluation.py`](../src/evaluation.py)) resamples the held-out set with replacement 1,000 times and reports how often each model comes out ahead — turning "0.849 vs. 1.125" into a claim about robustness, not just one number.

In [12]:
from evaluation import bootstrap_compare

ecfp_preds = xgb_ecfp.predict(X_test)
chemberta_preds = mlp_chemberta.predict(X_test_cb)

win_rates = bootstrap_compare(y_test, ecfp_preds, chemberta_preds, n_boot=1000, seed=0)
print("Fraction of 1,000 bootstrap resamples where XGBoost + ECFP beat MLP + ChemBERTa:")
for metric, win_rate in win_rates.items():
    print(f"  {metric}: {win_rate:.3f}")

Fraction of 1,000 bootstrap resamples where XGBoost + ECFP beat MLP + ChemBERTa:
  rmse: 1.000
  r2: 1.000
  spearman: 1.000


### Discussion: why did the simpler baseline win here?

XGBoost + ECFP beats MLP + ChemBERTa on RMSE, R², and Spearman ρ in essentially every bootstrap resample — this isn't a fragile, one-test-set artifact. A few concrete, non-mutually-exclusive reasons this particular comparison likely came out this way:

- **ChemBERTa is frozen, not fine-tuned.** It was pretrained on ~100M generic ZINC15 molecules for general chemical structure, not on kinase-inhibitor binding specifically. Without fine-tuning, its embeddings carry broad chemical-similarity signal but nothing tailored to *this* binding site — the MLP head has to extract task-specific structure-activity signal from a representation that wasn't built for the task.
- **Sample size favors the classical method.** ~4,452 training compounds is a small-to-medium dataset for training a neural network from a 768-dim representation; gradient-boosted trees on a sparse, high-dimensional binary fingerprint (2,048 bits) are a well-established strong, sample-efficient baseline in exactly this regime — which is precisely why Design Doc §5.3 called for including it rather than skipping straight to the embedding model.
- **ECFP substructure bits are close to directly interpretable pharmacophore signal** (specific substructure fragments known to matter for kinase binding), whereas a mean-pooled sentence-level embedding is a more diffuse, indirect encoding of the same molecule.

**Important scope caveat:** this result is specific to *this* setup — no hyperparameter tuning on either model, and ChemBERTa used strictly frozen rather than fine-tuned. It's evidence for "don't assume sophistication wins by default here," not a general claim that fingerprints beat pretrained embeddings on every problem or that fine-tuning ChemBERTa wouldn't close the gap.

## 6. Save trained models and evaluation metrics

Persist both fitted models (`results/models/`, via `joblib`) and the comparison table (`results/model_comparison.csv`) so later notebooks (e.g. Phase 5's selectivity analysis) can load the trained models directly instead of retraining, and so the metrics are available as a durable artifact rather than only living in this notebook's output cells.

Model files are reproducible by re-running this notebook (fixed `random_state`/seeds throughout) and are gitignored accordingly, matching the existing convention for `data/raw/` and `data/processed/`. `results/model_comparison.csv` is the actual small deliverable and is tracked in git.

In [13]:
import joblib

RESULTS_DIR = Path("../results")
MODELS_DIR = RESULTS_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(xgb_ecfp, MODELS_DIR / "xgb_ecfp.joblib")
joblib.dump(mlp_chemberta, MODELS_DIR / "mlp_chemberta.joblib")

print(f"Saved {MODELS_DIR / 'xgb_ecfp.joblib'}")
print(f"Saved {MODELS_DIR / 'mlp_chemberta.joblib'}")

Saved ../results/models/xgb_ecfp.joblib
Saved ../results/models/mlp_chemberta.joblib


### Verify the saved models actually reload correctly

Confirm the round-tripped models predict identically to the in-memory ones — not just that `joblib.dump` ran without an error.

In [14]:
reloaded_xgb = joblib.load(MODELS_DIR / "xgb_ecfp.joblib")
reloaded_mlp = joblib.load(MODELS_DIR / "mlp_chemberta.joblib")

assert np.array_equal(reloaded_xgb.predict(X_test), ecfp_preds), "Reloaded XGBoost model predicts differently!"
assert np.array_equal(reloaded_mlp.predict(X_test_cb), chemberta_preds), "Reloaded MLP model predicts differently!"
print("Both reloaded models reproduce the original in-memory predictions exactly.")

Both reloaded models reproduce the original in-memory predictions exactly.


### Write the comparison table to `results/`

Combines step 4's RMSE/R²/Spearman with step 5's bootstrap win-rates into one durable, trackable artifact.

In [15]:
comparison_table = metrics_df.copy()
comparison_table["bootstrap_win_rate_vs_other_model"] = [
    win_rates["rmse"],  # XGBoost + ECFP's win rate (computed as model A in step 5)
    1 - win_rates["rmse"],  # MLP + ChemBERTa's complementary win rate
]

comparison_path = RESULTS_DIR / "model_comparison.csv"
comparison_table.to_csv(comparison_path)
print(f"Saved {comparison_path}")
comparison_table

Saved ../results/model_comparison.csv


,RMSE,R²,Spearman ρ,bootstrap_win_rate_vs_other_model
XGBoost + ECFP,0.849183,0.519652,0.701822,1.0
MLP + ChemBERTa,1.125421,0.156309,0.551134,0.0


### Summary

**Step 1 — scaffold split:**
- 4,452 rows train / 1,113 rows test (80.0%/20.0% split), grouped by 814 train + 1,113 test Bemis-Murcko scaffolds rather than randomly.
- Zero scaffold overlap between train and test confirmed directly.
- Every compound's WT/D816V row pair confirmed to land entirely on one side of the split (never straddling), since they share identical scaffolds. In practice, all 925 compounds with both WT and D816V measurements happen to land in train under this split/seed — worth keeping in mind for Phase 5 (selectivity analysis), which relies on paired WT/D816V compounds.
- Split is seeded and deterministic (`seed=0`); a different seed changes which scaffold groups land in test while preserving the ~80/20 split size and the zero-leakage guarantee.

**Step 2 — XGBoost-on-ECFP baseline:**
- Trained on 4,452 compounds × 2,048-bit ECFP fingerprints, default hyperparameters (`src/models.py::train_xgboost_ecfp`).
- Sanity checks only: training R² = 0.809, held-out predictions non-degenerate (std = 0.841) and positively correlated with truth (Pearson r = 0.725).

**Step 3 — MLP-on-ChemBERTa comparison model:**
- Trained on 4,452 compounds × 768-dim frozen ChemBERTa embeddings (standardized first), default architecture, early-stopped at 42 iterations (`src/models.py::train_mlp_chemberta`).
- Sanity checks only: training R² = 0.748, held-out predictions non-degenerate (std = 1.174) and positively correlated with truth (Pearson r = 0.562).

**Step 4 — formal held-out evaluation (RMSE / R² / Spearman ρ):**

| Model | RMSE | R² | Spearman ρ |
| --- | --- | --- | --- |
| XGBoost + ECFP | 0.849 | 0.520 | 0.702 |
| MLP + ChemBERTa | 1.125 | 0.156 | 0.551 |

- Naive mean-baseline RMSE on the same held-out set: 1.233. Both models clearly beat it on RMSE and R² (verified directly, not just assumed).

**Step 5 — direct comparison:**
- Bootstrapped the held-out set 1,000 times (`src/evaluation.py::bootstrap_compare`): XGBoost + ECFP beat MLP + ChemBERTa on RMSE, R², *and* Spearman ρ in **100% of resamples** — a robust win, not a one-test-set artifact.
- Likely why (discussed in the notebook, with an explicit scope caveat): ChemBERTa is used frozen/not fine-tuned to this specific binding task, the dataset size (~4,452 train compounds) favors a sample-efficient tree ensemble over a neural net trained from scratch on top of embeddings, and ECFP substructure bits are closer to directly-relevant pharmacophore signal than a generic mean-pooled embedding. This is evidence for *this* setup, not a general claim that fingerprints beat pretrained embeddings, or that fine-tuning ChemBERTa wouldn't close the gap.

**Step 6 — saved artifacts:**
- Both fitted models saved to `results/models/` via `joblib` (gitignored, reproducible by re-running this notebook — fixed seeds throughout); reloading and re-predicting confirmed to match the in-memory models exactly, not just that the save call didn't error.
- Comparison table (RMSE/R²/Spearman ρ + bootstrap win rate) saved to `results/model_comparison.csv` — tracked in git as the actual durable deliverable.
- Next (not yet done in this pass): generate diagnostic plots (predicted vs. actual, residuals) to `results/figures/` (step 7).